In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Add the repository root to sys.path
repo_root = Path.cwd().resolve().parents[0]
if str(repo_root) not in sys.path:  sys.path.append(str(repo_root))

try:
    import torch
    import dataclasses
    import torch.nn as nn
    import torch.optim as optim
    import numpy as np
    import os
    import wandb
    from typing import List, Dict
    from moldiff.Dataset import PDBbind_Dataset
    from torch_geometric.loader import DataLoader
    from torch_geometric.data import Data
    from datetime import datetime

    # Import our molecular modules
    from moldiff.molecular_diffusion import (
        MolecularDenoisingModel, 
        exponential_noise_schedule,
        log_uniform_sampling,
        edm_weighting,
        MolecularDiffusion, 
        remove_mean_batch
    )
    from moldiff.metrics import (
        load_checkpoint,
        sample_molecules, 
        evaluate_atom_aa_distributions
    )
    from moldiff.egnn_dynamics import EGNNDynamics
    import torch.nn.functional as F

    from main import (
        CONFIG, 
        create_random_training_data, 
        create_random_eval_data, 
        create_schemes,
        create_molecular_model,
        train_model,
        save_checkpoint,
        analyze_samples
    )
    
except Exception as e:
    import traceback, sys
    print('Import failed:')
    print(e)
    traceback.print_exc()


In [ ]:
# set protein conditioning
CONFIG["protein_conditioning"] = True

# try:
# 1. Generate training and evaluation data
# print(f"\n{'='*60}")
# print("GENERATING TRAINING DATA")
# print(f"{'='*60}")

# train_data = create_batches_from_dataset(CONFIG['train_dataset_path'], CONFIG)
# eval_data = create_batches_from_dataset(CONFIG['eval_dataset_path'], CONFIG)
# create random data for demonstration
train_data = create_random_training_data(CONFIG)
eval_data = create_random_eval_data(CONFIG)

# Show example batch 
example_batch = train_data[0]
print(f"\nExample batch:")
print(f"  Ligand atoms: {example_batch['ligand_coords'].shape[0]}")
print(f"  Pocket residues: {example_batch['pocket_coords'].shape[0]}")
example_batch.keys()
# # 2. Create and initialize model
model = create_molecular_model(CONFIG)

# 3. Train the model
train_model(model, train_data, eval_data, CONFIG)


    


Generating 100 training batches...
✅ Generated 100 training batches
Generating 20 evaluation batches...
✅ Generated 20 evaluation batches

Example batch:
  Ligand atoms: 128
  Pocket residues: 372
Creating molecular diffusion model...
✅ Initialized MolecularDenoisingModel with 1141744 parameters


In [ ]:


# 4. Save checkpoint
save_checkpoint(model, CONFIG, save_path=CONFIG['checkpoint_path'].replace(".pt", "_final.pt"))

# 5. Load checkpoint to demonstrate persistence
print(f"\n{'='*60}")
print("LOADING MODEL FROM CHECKPOINT")
print(f"{'='*60}")

loaded_model = load_checkpoint(CONFIG['checkpoint_path'])

# 6. Sample new molecules
samples = sample_molecules(
    loaded_model,
    num_steps=CONFIG['num_sampling_steps'],
    schedule_type=CONFIG['schedule_type'],
    num_samples=CONFIG['num_eval_samples'],
)

for s in range(CONFIG['num_eval_samples']):
    graph = Data(
        ligand_coords=samples['ligand_coords'][samples['ligand_mask']==s].cpu(),
        ligand_features=samples['ligand_features'][samples['ligand_mask']==s].cpu(),
        pocket_coords=samples['pocket_coords'][samples['pocket_mask']==s].cpu(),
        pocket_features=samples['pocket_features'][samples['pocket_mask']==s].cpu()
    )
    torch.save(graph, CONFIG['run_dir'] + f"/sample_{s}.pt")


# 7. Analyze samples
analyze_samples(samples, CONFIG)

print(f"\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print(f"Checkpoint saved at: {CONFIG['checkpoint_path']}")
if CONFIG.get('device', 'cpu').startswith('cuda') and torch.cuda.is_available():
    print(f"Used {torch.cuda.get_device_name()} for computation")
else:
    print("Used CPU for computation")
    
# except Exception as e:
#     # Log error to wandb if it's initialized
#     if wandb.run is not None:
#         wandb.run.finish(exit_code=1)
#     raise e